In [7]:
import re
import pandas as pd
import pickle as pkl
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials

In [2]:
def extract_values(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    seed = int(file_path.split('_')[-1].split('.')[0])
    # Split the file into sections based on pruning labels
    sections = re.split(r'~~~(.*?)~~~', content)
    # Sections should be in the format: [text_before, title1, content1, title2, content2, ...]
    # So we'll extract pairs of (section title, section content)
    df = pd.DataFrame(columns = ['M', 'N', 'seed', 'match', 'crit_pt_lines_counted', 'time_crit_pts', 'exact_lines_counted', 'time_bneck', 'time_switch', 'time_match_dist', 'total_exact_time', 'prune'])
    for i in range(1, len(sections), 2):
        section_name = sections[i].strip()
        section_content = sections[i + 1]
        
        match = re.search(r'Rects in M:\s*([\d]+)', section_content)
        if match:
            M = int(match.group(1))
            
        match = re.search(r'Rects in N:\s*([\d]+)', section_content)
        if match:
            N = int(match.group(1))
        
        match = re.search(r'Maximum distance for lines through critical points:\s*([\d.]+)', section_content)
        if match:
            crit_point_dist = float(match.group(1))
            
        match = re.search(r'Maximum distance computed for grid of lines:\s*([\d.]+)', section_content)
        if match:
            grid_dist = float(match.group(1))
            
        match = re.search(r'Matching dist:\s*([\d.]+)', section_content)
        if match:
            exact_dist = float(match.group(1))
            
        if crit_point_dist == grid_dist == exact_dist:
            dist_match = 'yes'
        else:
            dist_match = f"crit_point_dist: {crit_point_dist}, grid_dist: {grid_dist}, exact_dist: {exact_dist}"
                
        match = re.search(r'Lines counted crit pts:\s*([\d]+)', section_content)
        if match:
            crit_pt_lines_counted = int(match.group(1))
        
        match = re.search(r'Time to get approx. from crit pts:\s*([\d.]+)', section_content)
        if match:
            time_crit_pts = float(match.group(1))
        
        match = re.search(r'Lines counted:\s*([\d]+)', section_content)
        if match:
             exact_lines_counted = int(match.group(1))
                
        match = re.search(r'Time to compute bottleneck dists:\s*([\d.]+)', section_content)
        if match:
            time_bneck = float(match.group(1))
            
        match = re.search(r'Time to compute switch pts:\s*([\d.]+)', section_content)
        if match:
            time_switch = float(match.group(1))
        
        match = re.search(r'Time to compute match dist:\s*([\d.]+)', section_content)
        if match:
            time_match_dist = float(match.group(1))
            
        match = re.search(r'Total time elapsed in exact computation:\s*([\d.]+)', section_content)
        if match:
            total_exact_time = float(match.group(1))
       
        row = pd.Series({'M': M, 'N': N, 'seed': seed, 'match': dist_match, 'crit_pt_lines_counted': crit_pt_lines_counted, 'time_crit_pts': time_crit_pts, 'exact_lines_counted': exact_lines_counted, 'time_bneck': time_bneck, 'time_switch': time_switch, 'time_match_dist': time_match_dist, 'total_exact_time': total_exact_time, 'prune': section_name})
        df = pd.concat([df, row.to_frame().T], ignore_index=True)
    return df

In [40]:
file_path = './all_param_combs/1_1_11.txt'
type(extract_values(file_path))

pandas.core.frame.DataFrame

### six rectangles in total

In [3]:
datasheet = pd.DataFrame()
seed = 1
pairs = [(1, 5), (5, 1), (2, 4), (4, 2), (3, 3)]
for M_num, N_num in pairs:
    file_path = f"./from_dardel/experiments_output/{M_num}_{N_num}_{seed}.txt"
    new_data = extract_values(file_path)
    datasheet = pd.concat([datasheet, new_data], ignore_index=True)
datasheet

,M,N,seed,match,crit_pt_lines_counted,time_crit_pts,exact_lines_counted,time_bneck,time_switch,time_match_dist,total_exact_time,prune
0,1,5,1,yes,1378,0.096,746558335,23549.853,90.25,27675.901,27766.151,NO PRUNING WHATSOEVER
1,1,5,1,yes,1378,0.127,133608828,4863.863,71.364,5401.04,5472.439,ONLY OLD PRUNING
2,1,5,1,yes,1378,0.142,50110154,1784.203,33.5,2014.138,2047.652,ALL (including NEW) PRUNING
3,5,1,1,yes,1378,0.111,737419802,32585.691,89.989,36882.236,36972.225,NO PRUNING WHATSOEVER
4,5,1,1,yes,1378,0.198,133762958,6756.161,71.696,7272.861,7344.587,ONLY OLD PRUNING
5,5,1,1,yes,1378,0.156,49595108,2403.094,34.633,2630.558,2665.209,ALL (including NEW) PRUNING
6,2,4,1,yes,1378,0.16,802312819,47212.228,88.965,51663.377,51752.344,NO PRUNING WHATSOEVER
7,2,4,1,yes,1378,0.39,140291313,9574.4,70.998,10250.3,10321.3,ONLY OLD PRUNING
8,2,4,1,yes,1378,0.174,55569316,3752.682,36.74,4000.458,4037.22,ALL (including NEW) PRUNING
9,4,2,1,yes,1378,0.137,795304945,36941.501,90.16,41412.022,41502.182,NO PRUNING WHATSOEVER


In [4]:
# save dataframe
with open('dardel_experiment_results_six.pkl', 'wb') as f:
    pkl.dump(datasheet, f)
datasheet.to_csv('dardel_experiment_results_six.csv', index=False)

In [ ]:
pairs = [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1), (1, 4), (4, 1), (2, 3), (3, 2)]


for x, y in pairs:
    print(x, y)

In [6]:
datasheet = pd.DataFrame()

pairs = [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1), (1, 4), (4, 1), (2, 3), (3, 2)]
for M_num, N_num in pairs:
    for seed_choice in range(20):
        file_path = f"./from_dardel/experiments_output/{M_num}_{N_num}_{seed_choice+1}.txt"
        new_data = extract_values(file_path)
        datasheet = pd.concat([datasheet, new_data], ignore_index=True)
datasheet

,M,N,seed,match,crit_pt_lines_counted,time_crit_pts,exact_lines_counted,time_bneck,time_switch,time_match_dist,total_exact_time,prune
0,1,1,1,yes,16,0.005,48721,0.546,0.723,0.772,1.495,NO PRUNING WHATSOEVER
1,1,1,1,yes,16,0.0,6507,0.084,0.473,0.104,0.577,ONLY OLD PRUNING
2,1,1,1,yes,16,0.0,2854,0.035,0.338,0.043,0.382,ALL (including NEW) PRUNING
3,1,1,2,yes,20,0.027,45932,0.645,0.703,0.852,1.555,NO PRUNING WHATSOEVER
4,1,1,2,yes,20,0.0,7053,0.114,0.463,0.133,0.596,ONLY OLD PRUNING
...,...,...,...,...,...,...,...,...,...,...,...,...
595,3,2,19,"crit_point_dist: 4.087, grid_dist: 4.241, exac...",702,0.046,26087315,1291.395,30.908,1408.97,1439.882,ONLY OLD PRUNING
596,3,2,19,"crit_point_dist: 4.087, grid_dist: 4.241, exac...",702,0.039,7833493,378.688,12.546,417.257,429.803,ALL (including NEW) PRUNING
597,3,2,20,yes,571,0.033,160807503,6411.811,40.117,7261.45,7301.567,NO PRUNING WHATSOEVER
598,3,2,20,yes,571,0.031,19275168,825.162,32.081,898.982,931.064,ONLY OLD PRUNING


In [7]:
# save dataframe
with open('dardel_experiment_results.pkl', 'wb') as f:
    pkl.dump(datasheet, f)
datasheet.to_csv('dardel_experiment_results.csv', index=False)

In [12]:
file_path = './all_param_combs/1_1_1.txt'
with open(file_path, 'r') as f:
    content = f.read()

# Split the file into sections based on pruning labels
sections = re.split(r'~~~(.*?)~~~', content)
section = sections[2]

'\n_____________________________________\nAPPROX\n_____________________________________\nRects in M:  1\nRects in N:  1\nMaximum distance for lines through critical points: 3.033\nLines counted crit pts: 16\nTime to get approx. from crit pts:  0.000 sec\nMaximum distance computed for grid of lines: 3.033\nNumber of grid points: 9800\nTime elapsed (theta b grid): 0.079 sec\n______________________________________\nEXACT\n_____________________________________\n3v1: Computed 79 pts; 32 deduplicated\n-------------\n2v2paired: Computed 11 pts; 10 deduplicated\n-------------\n2v2unpaired: Computed 30 pts; 28 deduplicated\n______________________________________\nMatching dist: 3.033 with slope (1.000, 0.997) and intercept (2.881, -2.881).\nLines counted: 2854\nTime to compute bottleneck dists: 0.024 sec\nTime to compute switch pts: 0.144 sec\nTime to compute match dist: 0.029 sec\nTotal time elapsed in exact computation: 0.173 sec\n_____________________________________\nEND\n__________________

In [20]:
i = 1
section_content = sections[i + 1]
section_content

'\n_____________________________________\nAPPROX\n_____________________________________\nRects in M:  1\nRects in N:  1\nMaximum distance for lines through critical points: 3.033\nLines counted crit pts: 16\nTime to get approx. from crit pts:  0.000 sec\nMaximum distance computed for grid of lines: 3.033\nNumber of grid points: 9800\nTime elapsed (theta b grid): 0.079 sec\n______________________________________\nEXACT\n_____________________________________\n3v1: Computed 562 pts; 162 deduplicated\n-------------\n2v2paired: Computed 20 pts; 17 deduplicated\n-------------\n2v2unpaired: Computed 255 pts; 105 deduplicated\n______________________________________\nMatching dist: 3.033 with slope (1.000, 0.997) and intercept (2.881, -2.881).\nLines counted: 48721\nTime to compute bottleneck dists: 0.354 sec\nTime to compute switch pts: 0.231 sec\nTime to compute match dist: 0.496 sec\nTotal time elapsed in exact computation: 0.726 sec\n_____________________________________\nEND\n_____________

In [23]:
match = re.search(r'Rects in N:\s*([\d]+)', section_content)
if match:
    N = int(match.group(1))
N

In [25]:
match = re.search(r'Matching dist:\s*([\d.]+)', section_content)
if match:
    exact_dist = float(match.group(1))
exact_dist

3.033

In [ ]:
# Each row as a Series
row1 = pd.Series({'name': 'Alice', 'age': 25})
row2 = pd.Series({'name': 'Bob', 'age': 30})
row3 = pd.Series({'name': 'Charlie', 'age': 35})

# Combine into DataFrame
df = pd.DataFrame([row1, row2, row3])
print(df)

In [7]:
from graphviz import Digraph

# Create directed graph
dot = Digraph(engine="neato")

# General node style
dot.attr("node", shape="circle", style="filled", fontname="Helvetica", fontsize="12")

# Nodes
dot.node("1", "1", fillcolor="#ffffff", color="red", penwidth="3")
dot.node("2", "2", fillcolor="#ffffff", color="green", penwidth="3")
dot.node("3", "3", fillcolor="#ffffff", color="black")
dot.node("4", "4", fillcolor="#ffffff", color="black")
dot.node("5", "5", fillcolor="#ffffff", color="black")
dot.node("6", "6", fillcolor="#ffffff", color="black")
dot.node("7", "7", fillcolor="#ffffff", color="black")
dot.node("8", "8", fillcolor="#ffffff", color="black")
dot.node("9", "9", fillcolor="#ffffff", color="black")

# Edges
dot.edge("3", "1", color="red")
dot.edge("4", "3", color="red")
dot.edge("7", "3", color="red")
dot.edge("6", "2", color="green")
dot.edge("7", "2", color="green")
dot.edge("3", "6", color="black")
dot.edge("6", "7", color="black")
dot.edge("2", "5", color="black")
dot.edge("2", "8", color="black")
dot.edge("5", "9", color="black")

# Labels as separate nodes with plaintext style
dot.node("label1", "basin of attraction", shape="plaintext", fontcolor="red", fontsize="14")
dot.node("label2", "basin of attraction", shape="plaintext", fontcolor="green", fontsize="14")
dot.node("label3", "boundary", shape="plaintext", fontcolor="gray", fontsize="14")
dot.node("label4", "index 0 critical node", shape="plaintext", fontcolor="red", fontsize="12")
dot.node("label5", "index 0 critical node", shape="plaintext", fontcolor="green", fontsize="12")

# Invisible edges to help position labels near the right nodes
dot.edge("label1", "1", style="invis")
dot.edge("label2", "2", style="invis")
dot.edge("label3", "6", style="invis")
dot.edge("label4", "1", style="invis")
dot.edge("label5", "2", style="invis")

# Export to files
dot.render("professional_schematic_graph_with_labels.pdf", format="pdf", cleanup=True)
dot.render("professional_schematic_graph_with_labels.png", format="png", cleanup=True)

print("Files saved as professional_schematic_graph_with_labels.pdf and .png")



Files saved as professional_schematic_graph_with_labels.pdf and .png


3

## get corner points

In [ ]:
def extract_corners(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    seed = int(file_path.split('_')[-1].split('.')[0])
    # Split the file into sections based on pruning labels
    sections = re.split(r'~~~(.*?)~~~', content)
    # Sections should be in the format: [text_before, title1, content1, title2, content2, ...]
    # So we'll extract pairs of (section title, section content)
    df = pd.DataFrame(columns = ['M', 'N', 'seed', 'match', 'crit_pt_lines_counted', 'time_crit_pts', 'exact_lines_counted', 'time_bneck', 'time_switch', 'time_match_dist', 'total_exact_time', 'prune'])
    for i in range(1, len(sections), 2):
        section_name = sections[i].strip()
        section_content = sections[i + 1]
        
        match = re.search(r'Rects in M:\s*([\d]+)', section_content)
        if match:
            M = int(match.group(1))
            
        match = re.search(r'Rects in N:\s*([\d]+)', section_content)
        if match:
            N = int(match.group(1))
        
        match = re.search(r'Maximum distance for lines through critical points:\s*([\d.]+)', section_content)
        if match:
            crit_point_dist = float(match.group(1))
            
        match = re.search(r'Maximum distance computed for grid of lines:\s*([\d.]+)', section_content)
        if match:
            grid_dist = float(match.group(1))
            
        match = re.search(r'Matching dist:\s*([\d.]+)', section_content)
        if match:
            exact_dist = float(match.group(1))
            
        if crit_point_dist == grid_dist == exact_dist:
            dist_match = 'yes'
        else:
            dist_match = f"crit_point_dist: {crit_point_dist}, grid_dist: {grid_dist}, exact_dist: {exact_dist}"
                
        match = re.search(r'Lines counted crit pts:\s*([\d]+)', section_content)
        if match:
            crit_pt_lines_counted = int(match.group(1))
        
        match = re.search(r'Time to get approx. from crit pts:\s*([\d.]+)', section_content)
        if match:
            time_crit_pts = float(match.group(1))
        
        match = re.search(r'Lines counted:\s*([\d]+)', section_content)
        if match:
             exact_lines_counted = int(match.group(1))
                
        match = re.search(r'Time to compute bottleneck dists:\s*([\d.]+)', section_content)
        if match:
            time_bneck = float(match.group(1))
            
        match = re.search(r'Time to compute switch pts:\s*([\d.]+)', section_content)
        if match:
            time_switch = float(match.group(1))
        
        match = re.search(r'Time to compute match dist:\s*([\d.]+)', section_content)
        if match:
            time_match_dist = float(match.group(1))
            
        match = re.search(r'Total time elapsed in exact computation:\s*([\d.]+)', section_content)
        if match:
            total_exact_time = float(match.group(1))
       
        row = pd.Series({'M': M, 'N': N, 'seed': seed, 'match': dist_match, 'crit_pt_lines_counted': crit_pt_lines_counted, 'time_crit_pts': time_crit_pts, 'exact_lines_counted': exact_lines_counted, 'time_bneck': time_bneck, 'time_switch': time_switch, 'time_match_dist': time_match_dist, 'total_exact_time': total_exact_time, 'prune': section_name})
        df = pd.concat([df, row.to_frame().T], ignore_index=True)
    return df